# Hexapawn — เกมกระดานพร้อม AI คู่ต่อสู้

โจทย์: สร้างเกม **Hexapawn** (กระดาน 3×3 เบี้ยฝ่ายละ 3 ตัว) พร้อม AI คู่ต่อสู้ที่ใช้:

1. **Minimax** — ค้นหาต้นไม้เกมและเลือกตาเดินที่ดีที่สุดโดยสมมติคู่แข่งเล่นดีที่สุดเช่นกัน
2. **Alpha-Beta Pruning** — ตัดกิ่งของต้นไม้เกมที่ไม่มีทางถูกเลือก เพื่อลดจำนวนโหนดที่ต้องสำรวจ
3. **Rule-based heuristic** — ฟังก์ชันประเมินตำแหน่งเมื่อค้นหาไม่ถึงจุดจบเกม และช่วยจัดลำดับการค้นหาให้ตัดกิ่งได้ดีขึ้น

**กติกา Hexapawn:** เบี้ยเดินตรงไปข้างหน้า 1 ช่องเมื่อว่าง และกินทแยงมุมได้เมื่อมีเบี้ยฝ่ายตรงข้าม
ฝ่ายที่พาเบี้ยไปถึงแถวสุดท้ายของอีกฝ่ายก่อน หรือฝ่ายที่ทำให้อีกฝ่าย "ไม่มีตาเดิน" จะเป็นผู้ชนะ

รันเซลล์ทั้งหมดตามลำดับด้านล่าง

## 1. โครงสร้างกระดานและการเดินหมาก

In [ ]:
# ---------- โครงสร้างกระดานและการเดินหมาก ----------
# state คือ string ยาว 9 ตัวอักษร: index = row*3 + col
# row 0 = แถวบนสุด (rank 3, จุดเริ่มของฝ่ายดำ 'b')
# row 2 = แถวล่างสุด (rank 1, จุดเริ่มของฝ่ายขาว 'w')
# 'w' = เบี้ยขาว, 'b' = เบี้ยดำ, '.' = ช่องว่าง

INITIAL_STATE = "bbb...www"

def algebraic(idx: int) -> str:
    """แปลง index (0-8) เป็นสัญกรณ์แบบหมากรุก เช่น 'a1', 'b3'"""
    col, row = idx % 3, idx // 3
    return "abc"[col] + str(3 - row)

def algebraic_to_idx(s: str) -> int:
    """แปลงสัญกรณ์ เช่น 'a1' กลับเป็น index (0-8)"""
    col = "abc".index(s[0])
    rank = int(s[1])
    row = 3 - rank
    return row * 3 + col

def generate_moves(state: str, color: str) -> list:
    """หาตาเดินที่ถูกกติกาทั้งหมดของ color ('w' หรือ 'b') ในตำแหน่ง state"""
    moves = []
    direction = -1 if color == 'w' else 1   # ขาวเดินขึ้น (row ลด), ดำเดินลง (row เพิ่ม)
    opponent = 'b' if color == 'w' else 'w'
    for idx in range(9):
        if state[idx] != color:
            continue
        row, col = divmod(idx, 3)
        nr = row + direction
        if nr < 0 or nr > 2:
            continue
        fwd = nr * 3 + col
        if state[fwd] == '.':                        # เดินตรงไปข้างหน้า
            moves.append({'from': idx, 'to': fwd, 'capture': False})
        for dc in (-1, 1):                            # กินทแยงมุม
            nc = col + dc
            if nc < 0 or nc > 2:
                continue
            cap = nr * 3 + nc
            if state[cap] == opponent:
                moves.append({'from': idx, 'to': cap, 'capture': True})
    return moves

def apply_move(state: str, move: dict) -> str:
    arr = list(state)
    arr[move['to']] = arr[move['from']]
    arr[move['from']] = '.'
    return ''.join(arr)

def check_game_over(state: str, next_player: str) -> dict:
    """เช็คว่าเกมจบหรือยัง: ถึงแถวเป้าหมาย หรือ next_player ไม่มีตาเดิน"""
    if any(state[c] == 'w' for c in range(0, 3)):
        return {'over': True, 'winner': 'w', 'reason': 'goal'}
    if any(state[c] == 'b' for c in range(6, 9)):
        return {'over': True, 'winner': 'b', 'reason': 'goal'}
    if not generate_moves(state, next_player):
        return {'over': True, 'winner': ('b' if next_player == 'w' else 'w'), 'reason': 'stuck'}
    return {'over': False}


## 2. ฮิวริสติกแบบ Rule-based (ใช้ประเมินตำแหน่ง + จัดลำดับการค้นหา)

In [ ]:
# ---------- ฮิวริสติกแบบ Rule-based ----------
# ใช้ประเมินตำแหน่งเมื่อค้นหาไปไม่ถึงจุดจบเกม (กรณีจำกัดความลึกในโหมดง่าย/ปานกลาง)
# ประกอบด้วย 4 กติกา:
#   1) material    : จำนวนเบี้ยที่เหลือ (ยิ่งเหลือมากกว่าคู่แข่งยิ่งดี)
#   2) advancement : ความคืบหน้าของเบี้ยแต่ละตัวเข้าใกล้แถวเป้าหมาย
#   3) mobility     : จำนวนตาเดินที่เป็นไปได้ (ยืดหยุ่นกว่า = ได้เปรียบ)
#   4) threats      : จำนวนการกินที่คุกคามได้ในตาถัดไป

def evaluate(state: str, for_color: str) -> int:
    opp_color = 'b' if for_color == 'w' else 'w'
    material = 0
    advancement = 0
    for idx in range(9):
        p = state[idx]
        if p == '.':
            continue
        row = idx // 3
        progress = (2 - row) if p == 'w' else row     # 0..2 ยิ่งมากยิ่งใกล้เป้าหมาย
        if p == for_color:
            material += 1
            advancement += progress
        else:
            material -= 1
            advancement -= progress

    my_moves = generate_moves(state, for_color)
    opp_moves = generate_moves(state, opp_color)
    mobility = len(my_moves) - len(opp_moves)
    threats = sum(m['capture'] for m in my_moves) - sum(m['capture'] for m in opp_moves)

    return material * 15 + advancement * 4 + mobility * 1 + threats * 3


def order_moves(moves: list) -> list:
    """เรียงลำดับตาเดินแบบ rule-based (ตากินก่อน) เพื่อช่วยให้ alpha-beta ตัดกิ่งได้ดีขึ้น"""
    return sorted(moves, key=lambda m: m['capture'], reverse=True)


## 3. Minimax + Alpha-Beta Pruning

In [ ]:
# ---------- Minimax + Alpha-Beta Pruning ----------
# ai_color = ฝ่ายที่เรา maximize คะแนนให้ (คือฝ่าย AI)
# stats เก็บสถิติ: nodes ที่สำรวจ, nodes ที่ถูกตัดทิ้ง (pruned)

def minimax(state: str, player_to_move: str, depth: int,
            alpha: float, beta: float, ai_color: str, stats: dict):
    stats['nodes'] += 1

    over = check_game_over(state, player_to_move)
    if over['over']:
        # ให้คะแนนสูง/ต่ำมาก ๆ และบวกด้วย depth ที่เหลือ เพื่อให้ AI เลือกทางชนะที่เร็วที่สุด
        if over['winner'] == ai_color:
            return 10000 + depth, None
        return -10000 - depth, None

    if depth == 0:
        return evaluate(state, ai_color), None

    moves = order_moves(generate_moves(state, player_to_move))
    maximizing = (player_to_move == ai_color)
    best_score = float('-inf') if maximizing else float('inf')
    best_move = moves[0]

    for i, m in enumerate(moves):
        new_state = apply_move(state, m)
        next_player = 'b' if player_to_move == 'w' else 'w'
        score, _ = minimax(new_state, next_player, depth - 1, alpha, beta, ai_color, stats)

        if maximizing:
            if score > best_score:
                best_score, best_move = score, m
            alpha = max(alpha, best_score)
        else:
            if score < best_score:
                best_score, best_move = score, m
            beta = min(beta, best_score)

        if beta <= alpha:
            stats['pruned'] += len(moves) - (i + 1)   # ตัดกิ่งที่เหลือทั้งหมดทิ้ง
            break

    return best_score, best_move


DIFFICULTY = {'ง่าย': 1, 'ปานกลาง': 3, 'เอาชนะไม่ได้': 15}


## 4. แสดงผลกระดานแบบข้อความ

In [ ]:
# ---------- แสดงผลกระดานแบบข้อความ ----------

SYMBOLS = {'w': '♙', 'b': '♟', '.': '.'}

def print_board(state: str):
    print("   a   b   c")
    for row in range(3):
        rank = 3 - row
        cells = [SYMBOLS[state[row * 3 + col]] for col in range(3)]
        print(f"{rank}  " + "   ".join(cells) + f"  {rank}")
    print("   a   b   c")


## 5. เล่นกับ AI (โหมดข้อความ)

พิมพ์ตาเดินในรูปแบบ `a1-a2` (จาก-ไป) หรือ `q` เพื่อออกจากเกม
ปรับระดับ AI ได้ที่พารามิเตอร์ `difficulty`: `'ง่าย'`, `'ปานกลาง'`, `'เอาชนะไม่ได้'`
และเลือกได้ว่าจะให้ AI เดินก่อนด้วย `ai_first=True`

In [ ]:
# ---------- โหมดเล่นแบบข้อความ (คน vs AI) ----------

def play_game(difficulty: str = 'ปานกลาง', ai_first: bool = False):
    """
    difficulty: 'ง่าย' (depth 1) / 'ปานกลาง' (depth 3) / 'เอาชนะไม่ได้' (depth 15)
    ai_first  : True ถ้าให้ AI เดินก่อน (AI จะถือฝ่ายขาว)
    พิมพ์ตาเดินแบบ 'a1-a2' (จาก-ไป) หรือ 'q' เพื่อออกจากเกม
    """
    depth = DIFFICULTY.get(difficulty, 3)
    human_color = 'b' if ai_first else 'w'
    ai_color = 'w' if ai_first else 'b'
    state = INITIAL_STATE
    turn = 'w'   # ฝ่ายขาวเดินก่อนเสมอตามกติกา

    print(f"เริ่มเกม! คุณ = {'ขาว (♙)' if human_color=='w' else 'ดำ (♟)'}, "
          f"AI = {'ขาว (♙)' if ai_color=='w' else 'ดำ (♟)'}, ระดับ: {difficulty}\n")

    while True:
        over = check_game_over(state, turn)
        if over['over']:
            print_board(state)
            winner_txt = 'คุณ' if over['winner'] == human_color else 'AI'
            reason_txt = 'เดินถึงแถวสุดท้ายสำเร็จ' if over['reason'] == 'goal' else 'อีกฝ่ายไม่มีตาเดินเหลือ'
            print(f"\n*** จบเกม: {winner_txt} ชนะ ({reason_txt}) ***")
            return

        if turn == human_color:
            print_board(state)
            legal = generate_moves(state, turn)
            raw = input("ตาเดินของคุณ (เช่น a1-a2, หรือ q เพื่อออก): ").strip()
            if raw.lower() == 'q':
                print("ออกจากเกม")
                return
            try:
                frm, to = raw.split('-')
                move = next(m for m in legal
                            if m['from'] == algebraic_to_idx(frm) and m['to'] == algebraic_to_idx(to))
            except (ValueError, StopIteration):
                print("ตาเดินไม่ถูกต้อง ลองใหม่อีกครั้ง\n")
                continue
            state = apply_move(state, move)
            print(f"คุณ: {algebraic(move['from'])}{'x' if move['capture'] else '-'}{algebraic(move['to'])}\n")
        else:
            stats = {'nodes': 0, 'pruned': 0}
            score, move = minimax(state, ai_color, depth, float('-inf'), float('inf'), ai_color, stats)
            state = apply_move(state, move)
            print(f"AI: {algebraic(move['from'])}{'x' if move['capture'] else '-'}{algebraic(move['to'])}  "
                  f"(สำรวจ {stats['nodes']} โหนด, ตัดทิ้ง {stats['pruned']} โหนด, คะแนน {score})\n")

        turn = 'b' if turn == 'w' else 'w'


In [ ]:
# เรียกใช้เพื่อเริ่มเล่น (ต้องรันแบบ interactive ใน Jupyter)
play_game(difficulty='ปานกลาง', ai_first=False)


## 6. สาธิต: AI เล่นกับ AI (ค้นหาเต็มรูปแบบ)

ให้ทั้งสองฝ่ายค้นหาลึกสุด (depth 15 ครอบคลุมทุกความเป็นไปได้ของเกมนี้) แล้วดูว่าใครชนะ

In [ ]:
# ---------- สาธิต: ให้ AI เล่นกับ AI แบบค้นหาเต็มรูปแบบ ----------

state = INITIAL_STATE
turn = 'w'
move_log = []

while True:
    over = check_game_over(state, turn)
    if over['over']:
        print(f"ผู้ชนะ: ฝ่าย {'ขาว' if over['winner']=='w' else 'ดำ'}  "
              f"(เหตุผล: {'ถึงแถวเป้าหมาย' if over['reason']=='goal' else 'อีกฝ่ายไม่มีตาเดิน'})")
        break
    stats = {'nodes': 0, 'pruned': 0}
    score, move = minimax(state, turn, 15, float('-inf'), float('inf'), turn, stats)
    move_log.append(f"{algebraic(move['from'])}{'x' if move['capture'] else '-'}{algebraic(move['to'])}")
    state = apply_move(state, move)
    turn = 'b' if turn == 'w' else 'w'

print("ลำดับการเดิน:", ' '.join(move_log))
print(f"จำนวนตาทั้งหมด: {len(move_log)}")


## 7. ทดสอบยืนยันว่า AI เล่นได้อย่างสมบูรณ์แบบ

รันเกม 300 ครั้งโดยให้ฝ่ายขาวเดินแบบสุ่ม และฝ่ายดำเป็น AI ที่ค้นหาเต็มรูปแบบ
ผลลัพธ์ควรออกมาว่า AI ชนะทุกเกม (ตรงกับทฤษฎีบทของ Hexapawn 3×3 ที่ผู้เดินคนที่สองมีกลยุทธ์ชนะเสมอ)

In [ ]:
# ---------- ทดสอบยืนยัน: AI (ค้นหาเต็มรูปแบบ, เป็นฝ่ายที่สอง) ต้องไม่แพ้เลย ----------
import random

def play_random_vs_ai(ai_depth: int = 15, seed: int = None) -> str:
    """ฝ่ายขาว (มือใหม่) เดินสุ่ม, ฝ่ายดำ (AI) ค้นหาเต็มรูปแบบ -> คืนค่าผู้ชนะ"""
    rng = random.Random(seed)
    state = INITIAL_STATE
    turn = 'w'
    for _ in range(50):
        over = check_game_over(state, turn)
        if over['over']:
            return over['winner']
        if turn == 'w':
            move = rng.choice(generate_moves(state, 'w'))
        else:
            stats = {'nodes': 0, 'pruned': 0}
            _, move = minimax(state, 'b', ai_depth, float('-inf'), float('inf'), 'b', stats)
        state = apply_move(state, move)
        turn = 'b' if turn == 'w' else 'w'
    return None

N = 300
results = [play_random_vs_ai(seed=i) for i in range(N)]
ai_wins = results.count('b')
print(f"เล่น {N} เกม: AI (ฝ่ายดำ, ค้นหาเต็มรูปแบบ) ชนะ {ai_wins}/{N} เกม")
assert ai_wins == N, "AI ควรชนะทุกเกมเมื่อค้นหาเต็มรูปแบบ (ตามทฤษฎีบทของ Hexapawn 3x3 ที่ผู้เดินคนที่สองมีกลยุทธ์ชนะเสมอ)"
print("ผ่านการทดสอบ: การค้นหาเต็มรูปแบบทำให้ AI เล่นได้อย่างสมบูรณ์แบบ (unbeatable)")


## 8. (ทางเลือกเสริม) กระดานแบบคลิกได้ด้วย ipywidgets

ต้องติดตั้งก่อน: `pip install ipywidgets` แล้วรันเซลล์ด้านล่าง (ต้องรันใน Jupyter ที่รองรับ widgets)

In [ ]:
# ---------- (ทางเลือกเสริม) กระดานแบบคลิกได้ด้วย ipywidgets ----------
# ต้องติดตั้งก่อน: pip install ipywidgets
# ใช้งาน: เรียก HexapawnGUI(difficulty='ปานกลาง', ai_first=False).display()

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    _HAS_WIDGETS = True
except ImportError:
    _HAS_WIDGETS = False


class HexapawnGUI:
    def __init__(self, difficulty: str = 'ปานกลาง', ai_first: bool = False):
        if not _HAS_WIDGETS:
            print("ต้องติดตั้ง ipywidgets ก่อน: pip install ipywidgets")
            return
        self.depth = DIFFICULTY.get(difficulty, 3)
        self.human_color = 'b' if ai_first else 'w'
        self.ai_color = 'w' if ai_first else 'b'
        self.state = INITIAL_STATE
        self.turn = 'w'
        self.selected = None
        self.game_over = False

        self.buttons = [widgets.Button(description='', layout=widgets.Layout(width='56px', height='56px'),
                                        style={'font_weight': 'bold'}) for _ in range(9)]
        for idx, btn in enumerate(self.buttons):
            btn.on_click(self._make_handler(idx))

        self.status = widgets.HTML()
        self.log = widgets.Output(layout=widgets.Layout(border='1px solid #ccc', height='140px', overflow_y='auto'))
        grid = widgets.GridBox(self.buttons, layout=widgets.Layout(grid_template_columns='repeat(3, 60px)'))
        reset_btn = widgets.Button(description='เกมใหม่')
        reset_btn.on_click(lambda _: self._reset())
        self.box = widgets.VBox([self.status, grid, reset_btn, self.log])

    def display(self):
        if not _HAS_WIDGETS:
            return
        self._render()
        display(self.box)

    def _reset(self):
        self.state = INITIAL_STATE
        self.turn = 'w'
        self.selected = None
        self.game_over = False
        with self.log:
            clear_output()
        self._render()

    def _make_handler(self, idx):
        def handler(_btn):
            self._on_click(idx)
        return handler

    def _on_click(self, idx):
        if self.game_over or self.turn != self.human_color:
            return
        piece = self.state[idx]
        if self.selected == idx:
            self.selected = None
            self._render()
            return
        if piece == self.human_color:
            self.selected = idx
            self._render()
            return
        if self.selected is not None:
            legal = [m for m in generate_moves(self.state, self.turn) if m['from'] == self.selected]
            move = next((m for m in legal if m['to'] == idx), None)
            if move:
                self._commit(move)

    def _commit(self, move):
        mover = self.turn
        self.state = apply_move(self.state, move)
        with self.log:
            who = 'คุณ' if mover == self.human_color else 'AI'
            print(f"{who}: {algebraic(move['from'])}{'x' if move['capture'] else '-'}{algebraic(move['to'])}")
        self.selected = None
        over = check_game_over(self.state, 'b' if mover == 'w' else 'w')
        if over['over']:
            self.game_over = True
            winner_txt = 'คุณ' if over['winner'] == self.human_color else 'AI'
            self.status.value = f"<b>จบเกม: {winner_txt} ชนะ!</b>"
            self._render()
            return
        self.turn = 'b' if mover == 'w' else 'w'
        self._render()
        if self.turn == self.ai_color:
            self._ai_move()

    def _ai_move(self):
        stats = {'nodes': 0, 'pruned': 0}
        score, move = minimax(self.state, self.ai_color, self.depth,
                               float('-inf'), float('inf'), self.ai_color, stats)
        with self.log:
            print(f"AI คิด: สำรวจ {stats['nodes']} โหนด, ตัดทิ้ง {stats['pruned']} โหนด, คะแนน {score}")
        self._commit(move)

    def _render(self):
        for idx, btn in enumerate(self.buttons):
            piece = self.state[idx]
            btn.description = SYMBOLS[piece] if piece != '.' else ''
            if self.selected == idx:
                btn.button_style = 'warning'
            elif self.selected is not None:
                legal = [m for m in generate_moves(self.state, self.turn) if m['from'] == self.selected]
                btn.button_style = 'info' if any(m['to'] == idx for m in legal) else ''
            else:
                btn.button_style = ''
        if not self.game_over:
            who = 'คุณ' if self.turn == self.human_color else 'AI'
            self.status.value = f"ตาของ: <b>{who}</b> ({'♙' if self.turn=='w' else '♟'})"


In [ ]:
# ตัวอย่างการเรียกใช้ GUI (uncomment เพื่อรัน)
# HexapawnGUI(difficulty='ปานกลาง', ai_first=False).display()
